In [ ]:
import re
import math
import json

# ==========================================
# 1. SETTINGS & WEIGHTS
# ==========================================
EARNINGS_WEIGHT = 0.70
WINS_WEIGHT = 0.30

EASY_PERCENTILE = 0.10
MEDIUM_PERCENTILE = 0.40
# HARD is the remaining 40-100%

PLAYERS_FILE = 'players.ts'
ENTRIES_FILE = 'entries.ts'

# ==========================================
# 2. PARSE TYPESCRIPT FILES
# ==========================================
def parse_data():
    players = {}

    # Parse players.ts
    with open(PLAYERS_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            if "id:" in line and "name:" in line:
                pid = re.search(r"id:\s*'([^']+)'", line).group(1)
                name = re.search(r"name:\s*'([^']+)'", line).group(1)

                earnings_match = re.search(r"earnings:\s*(\d+)", line)
                known_match = re.search(r"earningsKnown:\s*(true|false)", line)

                players[pid] = {
                    'id': pid,
                    'name': name,
                    'earnings': int(earnings_match.group(1)) if earnings_match else 0,
                    'earnings_known': True if known_match and known_match.group(1) == 'true' else False,
                    'major_wins': 0,
                    'win_events': []
                }

    # Parse entries.ts
    with open(ENTRIES_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            if "eventId:" in line and "placement:" in line:
                event_id = re.search(r"eventId:\s*'([^']+)'", line).group(1)
                placement = int(re.search(r"placement:\s*(\d+)", line).group(1))

                if placement == 1:
                    players_match = re.search(r"playerIds:\s*\[(.*?)\]", line)
                    if players_match:
                        # Extract individual players from the array string e.g., 'aqua', 'tschinken'
                        pids = re.findall(r"'([^']+)'", players_match.group(1))
                        for pid in pids:
                            if pid in players:
                                players[pid]['major_wins'] += 1
                                players[pid]['win_events'].append(event_id)

    return players

# ==========================================
# 3. CALCULATE FAME SCORES
# ==========================================
def calculate_fame(players):
    # --- A. Earnings Score (Log10 Normalized) ---
    known_earnings = [
        math.log10(p['earnings'])
        for p in players.values()
        if p['earnings_known'] and p['earnings'] > 0
    ]

    max_log_e = max(known_earnings) if known_earnings else 1
    min_log_e = min(known_earnings) if known_earnings else 0

    # Calculate a neutral baseline for players with unknown earnings.
    # The 25th percentile of known log earnings prevents them from auto-sinking to 0.
    sorted_logs = sorted(known_earnings)
    unknown_baseline = sorted_logs[int(len(sorted_logs) * 0.25)] if sorted_logs else 0

    for p in players.values():
        if p['earnings_known'] and p['earnings'] > 0:
            raw_log = math.log10(p['earnings'])
        else:
            raw_log = unknown_baseline

        # Normalize to 0.0 - 1.0
        p['earnings_score'] = (raw_log - min_log_e) / (max_log_e - min_log_e) if max_log_e > min_log_e else 0

    # --- B. Major Wins Score (Normalized) ---
    max_wins = max([p['major_wins'] for p in players.values()]) if players else 1

    for p in players.values():
        p['wins_score'] = p['major_wins'] / max_wins

    # --- C. Final Fame Combine ---
    for p in players.values():
        fame_raw = (EARNINGS_WEIGHT * p['earnings_score']) + (WINS_WEIGHT * p['wins_score'])
        p['fame_score'] = fame_raw * 100  # Scale 0-100 for readability

    # Sort descending by Fame Score
    return sorted(players.values(), key=lambda x: x['fame_score'], reverse=True)

# ==========================================
# 4. ASSIGN TIERS & PRINT REPORT
# ==========================================
def print_report(sorted_players):
    total = len(sorted_players)
    easy_cutoff = int(total * EASY_PERCENTILE)
    medium_cutoff = int(total * MEDIUM_PERCENTILE)

    print("=== TIER BOUNDARIES ===")
    print(f"EASY   (Top {int(EASY_PERCENTILE*100)}%)  -> Rank #1 – #{easy_cutoff}")
    print(f"MEDIUM ({int(EASY_PERCENTILE*100)}–{int(MEDIUM_PERCENTILE*100)}%) -> Rank #{easy_cutoff + 1} – #{medium_cutoff}")
    print(f"HARD   ({int(MEDIUM_PERCENTILE*100)}–100%) -> Rank #{medium_cutoff + 1} – #{total}\n")

    print(f"Algorithm Weights: {int(EARNINGS_WEIGHT*100)}% Earnings / {int(WINS_WEIGHT*100)}% Major Wins\n")
    print("=== FAME RANKING (TOP 100 INSPECTION) ===")

    for i, p in enumerate(sorted_players[:100]):
        rank = i + 1

        # Determine Tier
        if rank <= easy_cutoff:
            tier = "⭐ Easy"
        elif rank <= medium_cutoff:
            tier = "🟡 Medium"
        else:
            tier = "🔴 Hard"

        win_str = ", ".join(p['win_events']) if p['win_events'] else "None"
        earnings_str = f"${p['earnings']:,}" if p['earnings_known'] else "Unknown"

        print(f"#{rank} {p['name']}  [{tier}]")
        print(f"   Fame:       {p['fame_score']:.2f}")
        print(f"   Earnings:   {earnings_str}")
        print(f"   Major wins: {p['major_wins']}")
        print(f"   Wins:       {win_str}\n")

    # Optional: Generate the fame-ranking.json file for reference
    export_data = [
        {
            "playerId": p['id'],
            "fameScore": round(p['fame_score'] / 100, 3), # Store as 0-1 internally
            "famePercentile": round((i + 1) / total, 3),
            "tier": "easy" if (i + 1) <= easy_cutoff else "medium" if (i + 1) <= medium_cutoff else "hard"
        }
        for i, p in enumerate(sorted_players)
    ]

    with open("fame-ranking.json", "w") as f:
        json.dump(export_data, f, indent=2)
    print("Exported complete tier list to 'fame-ranking.json'")

if __name__ == "__main__":
    players_dict = parse_data()
    ranked_players = calculate_fame(players_dict)
    print_report(ranked_players)